# Entrenar la wake word "oye deepy" (se pronuncia "oye dipi")

Este notebook entrena un modelo de **openWakeWord** para detectar la frase hablada
**"oye dipi"** (así se pronuncia el nombre del asistente, "Deepy") y producir un archivo
`oye_deepy.onnx` de ~200 KB que corre en CPU en tu PC. El nombre del archivo usa la
grafía "deepy" solo por consistencia con el resto del proyecto; lo que el modelo
realmente aprende a reconocer es el sonido "dipi".

**Nota importante:** "oye deepy" (leído letra por letra en español) y "oye dipi" (como se
pronuncia en la práctica) NO suenan igual — se verificó con el fonemizador de Piper que
"deepy" se lee "de-e-pi" (3 sílabas) mientras que "dipi" es "di-pi" (2 sílabas). Por eso el
texto que se sintetiza como positivo es "oye dipi", y "oye deepy" pasa a ser uno de los
negativos (una forma parecida que el modelo debe aprender a NO aceptar).

**Antes de empezar:**
1. Arriba a la derecha, en "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución",
   elige **GPU (T4)**.
2. Ejecuta las celdas en orden, de arriba hacia abajo.
3. Tarda entre 1 y 2 horas en total (la mayor parte es descarga de datos y entrenamiento).
4. Al final vas a descargar `oye_deepy.onnx` y lo vas a guardar en
   `voz_local/voces/wakeword/oye_deepy.onnx` en tu PC.

**Nota sobre el enfoque:** el pipeline "automático" oficial de openWakeWord genera las
frases de ejemplo con un motor de voz en inglés. Como "oye deepy" es una frase en español,
este notebook genera esos ejemplos con **voces de Piper en español** en su lugar, y
reutiliza el resto del pipeline oficial (augmentación, extracción de características,
entrenamiento, exportación a ONNX) tal cual.

## 1. Verificar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Instalar dependencias

Instalamos solo lo necesario para generar clips con Piper y entrenar con openWakeWord.
**No** instalamos TensorFlow ni las librerías de conversión a `.tflite`: nuestro
`asistente.py` solo necesita el modelo `.onnx`, así que nos salteamos esa parte
(más liviano y con muchas menos versiones fijadas y frágiles de instalar).

In [ ]:
# openWakeWord (clon editable, para tener acceso a train.py/data.py/utils.py)
!git clone https://github.com/dscripka/openWakeWord
!pip install -e ./openWakeWord --quiet

# Generador de muestras con Piper (paquete actual, usa piper-tts como motor)
!pip install piper-sample-generator --quiet

# piper-sample-generator importa (sin usarlo en el modo de voces normales que usamos
# acá) el módulo `piper_train.vits.commons` del repo de entrenamiento de Piper, que no
# está en PyPI como paquete instalable y cuyo propio requirements.txt fija `torch<2`
# (choca con el `torch>=2` que ya necesita piper-sample-generator). Por eso NO se instala
# con pip: se clona el repo y se agrega src/python al PYTHONPATH, así el import
# encuentra el módulo sin arrastrar ese requirements.txt conflictivo.
!git clone --depth 1 https://github.com/rhasspy/piper piper_repo --quiet
import os
os.environ["PYTHONPATH"] = os.path.abspath("piper_repo/src/python") + os.pathsep + os.environ.get("PYTHONPATH", "")

# Librerías que usa train.py para augmentar/entrenar (sin TensorFlow ni tflite,
# esas solo hacen falta para exportar a .tflite, que no usamos)
# datasets se fija a 3.6.0: desde la 4.0 el feature Audio decodifica con torchcodec
# (devuelve un AudioDecoder, no el dict {"array":..., "path":...} que usan las celdas de abajo)
!pip install torchinfo==1.8.0 torchmetrics==1.2.0 torch-audiomentations==0.11.0 pronouncing "datasets==3.6.0" --quiet
!pip install speechbrain==0.5.14 mutagen==1.47.0 acoustics==0.2.6 --quiet

# Parche directo sobre el archivo instalado de torch_audiomentations: la versión nueva
# de torchaudio (la de Colab) rompe set_audio_backend/info/load que este paquete llama
# de forma obligatoria. Un shim vía sitecustomize.py (más abajo) no siempre alcanza —
# torchaudio nuevo expone estos nombres como stubs deprecados que pasan hasattr() pero
# explotan al llamarlos — así que se edita el archivo instalado directamente, en los
# 3 puntos exactos donde llama a torchaudio, reemplazándolos por soundfile (ya
# instalado, misma librería que ya usa piper-tts en el proyecto).
import importlib.util
import re
from pathlib import Path

_spec = importlib.util.find_spec("torch_audiomentations")
_io_path = Path(_spec.origin).parent / "utils" / "io.py"
_texto = _io_path.read_text()
_texto_original = _texto

_texto = _texto.replace(
    "import torchaudio",
    "import torchaudio\nimport soundfile as _sf_shim",
    1,
)
_texto = _texto.replace(
    'torchaudio.set_audio_backend("soundfile")',
    'getattr(torchaudio, "set_audio_backend", lambda *a, **k: None)("soundfile")',
)
_texto = _texto.replace(
    "info = torchaudio.info(file_path)",
    "_meta_shim = _sf_shim.info(file_path)\n"
    '        info = type("_InfoShim", (), {"num_frames": _meta_shim.frames, "sample_rate": _meta_shim.samplerate})()',
)
# Reemplazo por regex (no texto literal): la llamada a torchaudio.load real está
# formateada en varias líneas y un match exacto de espacios es frágil — si no
# coincidiera, .replace() fallaría en silencio y el error volvería a aparecer más tarde.
_patron_load = re.compile(
    r"torchaudio\.load\(\s*audio_path,\s*frame_offset=original_sample_offset,\s*num_frames=original_num_samples,\s*\)"
)
_texto, _n_load = _patron_load.subn(
    "_sf_shim_read(audio_path, original_sample_offset, original_num_samples)",
    _texto,
)
_texto = _texto.replace(
    "class Audio:",
    'def _sf_shim_read(path, offset, num):\n'
    '    datos, sr = _sf_shim.read(str(path), start=offset, frames=num, dtype="float32", always_2d=True)\n'
    '    return torch.from_numpy(datos.T), sr\n'
    "\n\nclass Audio:",
    1,
)
assert _n_load == 1, "no se encontró la llamada a torchaudio.load esperada — revisar io.py instalado a mano"
_io_path.write_text(_texto)
print(f"{_io_path} parcheado (set_audio_backend, info, load).")

# speechbrain==0.5.14 y torch-audiomentations==0.11.0 (ambos import obligatorio de
# openwakeword/data.py) llaman funciones de torchaudio que las versiones nuevas de
# torchaudio (las que trae Colab por defecto) ya eliminaron: set_audio_backend, load,
# info, list_audio_backends. En vez de bajar la versión de torch/torchaudio (lento y
# arriesgado — Colab los tiene atados a la GPU disponible), se agregan como shims basados
# en soundfile via sitecustomize.py, que Python carga automáticamente en cada proceso
# nuevo — así el entrenamiento (que corre en un subproceso aparte) también los ve.
import os
import site
from pathlib import Path

# Ojo: NO se usa "if not hasattr(...)" para decidir si parchear. La versión nueva de
# torchaudio expone info/load/list_audio_backends como stubs deprecados que SÍ pasan
# hasattr() pero explotan al llamarlos (con este mismo AttributeError, de forma confusa)
# -> hay que sobreescribirlos siempre, sin condición.
_lineas_sitecustomize = [
    "import torch",
    "import torchaudio",
    "import soundfile as _sf",
    "",
    "torchaudio.set_audio_backend = lambda *a, **k: None",
    'torchaudio.list_audio_backends = lambda: ["soundfile"]',
    "",
    "class _Info:",
    "    def __init__(self, sample_rate, num_frames):",
    "        self.sample_rate = sample_rate",
    "        self.num_frames = num_frames",
    "",
    "def _info(path, *a, **k):",
    "    meta = _sf.info(str(path))",
    "    return _Info(meta.samplerate, meta.frames)",
    "",
    "torchaudio.info = _info",
    "",
    "def _load(path, *a, **k):",
    '    datos, sr = _sf.read(str(path), dtype="float32", always_2d=True)',
    "    return torch.from_numpy(datos.T), sr",
    "",
    "torchaudio.load = _load",
    "",
]
_destino = Path(site.getsitepackages()[0]) / "sitecustomize.py"
_destino.write_text("\n".join(_lineas_sitecustomize))
print(f"Shims de torchaudio escritos en {_destino}")

import sys, uuid
import numpy as np
import scipy.io.wavfile
import yaml
from tqdm import tqdm

print("Listo.")

## 3. Descargar los modelos base de openWakeWord

`melspectrogram.onnx` y `embedding_model.onnx` son los preprocesadores que openWakeWord
necesita siempre, además del modelo de la wake word en sí.

In [ ]:
import os
os.makedirs("./openWakeWord/openwakeword/resources/models", exist_ok=True)
!wget -q -O ./openWakeWord/openwakeword/resources/models/embedding_model.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx
!wget -q -O ./openWakeWord/openwakeword/resources/models/melspectrogram.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx
print("Modelos base descargados.")

## 4. Descargar datos de augmentación y negativos

Estos datos no dependen del idioma de la wake word: son ruido de fondo, respuestas de
impulso de sala (para simular eco/acústica) y ~2000 horas de audio ya convertido a
"features" de openWakeWord para actuar como ejemplos negativos genéricos durante el
entrenamiento. Es data equivalente a la que usa el notebook oficial (con AudioSet como
única fuente de ruido de fondo, en vez de AudioSet + Free Music Archive: el dataset de
FMA que usaba el notebook oficial requiere un script de carga personalizado que
actualmente falla al transmitir por HTTP; AudioSet solo ya da suficiente diversidad
para este modelo personal).

In [ ]:
# Respuestas de impulso de sala (MIT)
import datasets

output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Ruido/musica/voces de fondo: subconjunto "balanced" de AudioSet
# (el dataset ya no se distribuye como .tar — se cargan directamente los parquet
# oficiales con el loader de datasets, más robusto que descargar un archivo a mano)
output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)

audioset_dataset = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

N_CLIPS_FONDO = 3000  # alcanza para este modelo personal; si quieres más diversidad, súbelo
for i, row in enumerate(tqdm(audioset_dataset, total=N_CLIPS_FONDO, desc="AudioSet -> 16kHz")):
    if i >= N_CLIPS_FONDO:
        break
    name = f"{row['video_id']}.wav"
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Features pre-calculadas de openWakeWord: negativos genéricos (~2000h) y validación (~11h)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Features de negativos/validación descargadas.")

## 5. Descargar voces de Piper en español

Usamos 4 voces oficiales de Piper (2 de España, 2 de México) para que el modelo aprenda
a reconocer "oye deepy" dicho con distintos acentos y timbres, no memorice una sola voz.

In [ ]:
VOCES = [
    ("es_ES", "sharvard", "medium"),
    ("es_ES", "davefx", "medium"),
    ("es_MX", "claude", "high"),
    ("es_MX", "ald", "medium"),
]

os.makedirs("voces_es", exist_ok=True)
voice_paths = []
for lang, name, quality in VOCES:
    base = f"{lang}-{name}-{quality}"
    url_base = f"https://huggingface.co/rhasspy/piper-voices/resolve/main/{lang[:2]}/{lang}/{name}/{quality}/{base}"
    onnx_path = f"voces_es/{base}.onnx"
    if not os.path.exists(onnx_path):
        get_ipython().system(f"wget -q -O {onnx_path} {url_base}.onnx")
        get_ipython().system(f"wget -q -O {onnx_path}.json {url_base}.onnx.json")
    voice_paths.append(onnx_path)

print("Voces descargadas:", voice_paths)

## 6. (Opcional pero recomendado) Subir tus propias grabaciones

Grabaciones reales tuyas diciendo "oye deepy" mejoran mucho la detección de tu voz real
frente a solo usar voces sintéticas. Graba ~20-30 veces la frase "oye deepy" (variando
distancia al micrófono y tono), cualquier formato (wav/mp3/m4a), y súbelas aquí.

Si prefieres saltarte este paso por ahora, simplemente no corras la celda de subida — el
modelo se entrena igual solo con voces sintéticas, pero va a reconocer mejor tu voz si
agregas tus propias grabaciones.

In [ ]:
from google.colab import files

os.makedirs("mis_grabaciones_raw", exist_ok=True)
subidos = files.upload()
for nombre in subidos:
    os.rename(nombre, os.path.join("mis_grabaciones_raw", nombre))
print(f"{len(subidos)} grabaciones subidas.")

## 7. Generar clips sintéticos positivos ("oye dipi")

Usamos `piper-sample-generator` (el mismo motor Piper que ya usas en `leer_notas.py`,
pero en modo generador de muestras) ciclando las 4 voces descargadas, con pequeñas
variaciones de velocidad para dar diversidad. El texto que se sintetiza es "oye dipi"
(la pronunciación real), no "oye deepy" (ver nota al principio del notebook).

In [ ]:
FRASE = "oye dipi"
MODEL_NAME = "oye_deepy"  # nombre del archivo del modelo, por consistencia con el resto del proyecto

N_TRAIN = 4000
N_TEST = 400

def generar(texto_o_lista, n, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    textos = texto_o_lista if isinstance(texto_o_lista, list) else [texto_o_lista]
    modelos_args = " ".join(f"--model {p}" for p in voice_paths)
    for texto in textos:
        n_por_texto = max(1, n // len(textos))
        cmd = f'python3 -m piper_sample_generator "{texto}" {modelos_args} --max-samples {n_por_texto} --length-scales 0.9 1.0 1.1 --output-dir {out_dir}'
        get_ipython().system(cmd)

generar(FRASE, N_TRAIN, "raw_positive_train")
generar(FRASE, N_TEST, "raw_positive_test")
print("train:", len(os.listdir("raw_positive_train")), "| test:", len(os.listdir("raw_positive_test")))

## 8. Generar clips negativos (confusores en español)

En vez del generador automático de "negativos adversariales" (pensado para inglés, usa un
diccionario fonético en inglés que no sirve para español), usamos una lista escrita a mano
de frases parecidas a "oye dipi" para que el modelo aprenda a NO dispararse con ellas.
Se verificó con el fonemizador de Piper que ninguna de estas coincide fonéticamente con
"oye dipi" — en particular, "oye deepy" y "oye deepi" suenan idéntico entre sí
("de-e-pi") pero distinto de "dipi", así que son negativos legítimos, no el objetivo.

In [ ]:
NEGATIVOS = [
    "oye", "deepy", "oye deepy", "oye deepi", "hoy depi", "oye leidy",
    "oye felipe", "voy deprisa", "oye kevin", "oye betty", "soy deepy",
    "hola deepy", "eh deepy", "oye edy", "oye desi",
]

generar(NEGATIVOS, N_TRAIN, "raw_negative_train")
generar(NEGATIVOS, N_TEST, "raw_negative_test")
print("train:", len(os.listdir("raw_negative_train")), "| test:", len(os.listdir("raw_negative_test")))

## 9. Augmentar y resamplear a 16 kHz

`piper_sample_generator.augment` baja los clips de 22050 Hz (salida nativa de las voces
Piper que usamos, calidad medium/high) a los 16 kHz que espera openWakeWord, y de paso les
aplica variaciones de volumen y de acústica (eco/sala) para dar más robustez.

**Ojo con `--sample-rate`:** el nombre sugiere "la frecuencia de origen", pero en
realidad es la frecuencia **de salida** que le pedimos al conversor (mirando el código
fuente: solo resamplea si el valor pedido es distinto al de origen). Hay que pasar
`--sample-rate 16000` (el destino que necesitamos), no 22050 (el origen) — si se pasa
22050 el conversor ve que ya está en 22050 y no hace nada, dejando los clips sin
resamplear.

In [ ]:
FINAL_DIR = f"my_custom_model/{MODEL_NAME}"
os.makedirs(f"{FINAL_DIR}/positive_train", exist_ok=True)
os.makedirs(f"{FINAL_DIR}/positive_test", exist_ok=True)
os.makedirs(f"{FINAL_DIR}/negative_train", exist_ok=True)
os.makedirs(f"{FINAL_DIR}/negative_test", exist_ok=True)

get_ipython().system(f"python3 -m piper_sample_generator.augment --sample-rate 16000 raw_positive_train/ {FINAL_DIR}/positive_train/")
get_ipython().system(f"python3 -m piper_sample_generator.augment --sample-rate 16000 raw_positive_test/ {FINAL_DIR}/positive_test/")
get_ipython().system(f"python3 -m piper_sample_generator.augment --sample-rate 16000 raw_negative_train/ {FINAL_DIR}/negative_train/")
get_ipython().system(f"python3 -m piper_sample_generator.augment --sample-rate 16000 raw_negative_test/ {FINAL_DIR}/negative_test/")

import wave

for d in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    archivos = list(Path(f"{FINAL_DIR}/{d}").glob("*.wav"))
    print(f"{d}: {len(archivos)} clips")
    assert archivos, f"{d} quedó vacío — revisa la celda anterior antes de seguir"
    with wave.open(str(archivos[0]), "rb") as wf:
        assert wf.getframerate() == 16000, (
            f"{archivos[0]} está a {wf.getframerate()} Hz, no a 16000 Hz — revisa el --sample-rate de arriba"
        )

## 10. Mezclar tus grabaciones reales (si subiste alguna en el paso 6)

Se convierten con `ffmpeg` (ya viene instalado en Colab) a 16 kHz mono, el formato que
usa openWakeWord — `ffmpeg` acepta cualquier formato de origen (m4a, mp3, wav, ogg...),
así que no importa en qué formato hayas grabado. Se reparten 90%/10% entre entrenamiento
y test, igual que los clips sintéticos.

In [ ]:
import subprocess

raw_dir = "mis_grabaciones_raw"
extensiones_audio = {".wav", ".mp3", ".m4a", ".ogg", ".flac", ".aac", ".webm"}
archivos = sorted(
    p for p in Path(raw_dir).glob("*") if p.is_file() and p.suffix.lower() in extensiones_audio
) if os.path.exists(raw_dir) else []

if not archivos:
    print("No hay grabaciones propias subidas, se sigue solo con datos sintéticos.")
else:
    n_test = max(1, len(archivos) // 10)
    for i, archivo in enumerate(archivos):
        destino = "positive_test" if i < n_test else "positive_train"
        salida = f"{FINAL_DIR}/{destino}/mia_{i}.wav"
        subprocess.run(
            ["ffmpeg", "-y", "-i", str(archivo), "-ar", "16000", "-ac", "1", "-sample_fmt", "s16", salida],
            check=True, capture_output=True,
        )
    print(f"{len(archivos)} grabaciones propias agregadas ({n_test} a test, {len(archivos)-n_test} a train).")

## 11. Configuración de entrenamiento

Partimos del YAML de ejemplo oficial y solo ajustamos lo necesario. `n_samples` /
`n_samples_val` reflejan la cantidad real que generamos arriba, no un valor por defecto
mayor que no corresponde a los datos disponibles.

In [ ]:
config = yaml.load(open("openWakeWord/examples/custom_model.yml").read(), yaml.Loader)

config["model_name"] = MODEL_NAME
config["target_phrase"] = [FRASE]
config["n_samples"] = N_TRAIN
config["n_samples_val"] = N_TEST
config["steps"] = 10000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.4

# train.py importa `generate_samples` de este path de forma incondicional al arrancar,
# incluso si nunca se pasa --generate_clips (no está detrás de ese chequeo). Como ya
# generamos los clips a mano en los pasos 7-10, se apunta a un stub que satisface el
# import sin instalar la interfaz vieja de piper-sample-generator (rota, ver paso 7-9).
_stub_dir = Path("generate_samples_stub")
_stub_dir.mkdir(exist_ok=True)
(_stub_dir / "generate_samples.py").write_text(
    "def generate_samples(*args, **kwargs):\n"
    "    raise NotImplementedError('no se usa: los clips ya se generaron a mano en los pasos 7-10')\n"
)
config["piper_sample_generator_path"] = str(_stub_dir)
config["output_dir"] = "./my_custom_model"
config["rir_paths"] = ["./mit_rirs"]
config["background_paths"] = ["./audioset_16k"]
config["background_paths_duplication_rate"] = [1]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open("oye_deepy.yaml", "w") as f:
    yaml.dump(config, f)

print("Config guardada en oye_deepy.yaml")

## 12. Entrenar

Corremos solo `--augment_clips` y `--train_model` — nos saltamos `--generate_clips`
porque ya generamos y ubicamos los clips nosotros mismos en los pasos 7-10 (el generador
automático de `train.py` asume un motor de voz en inglés que no sirve para "oye deepy").

Esta celda es la que más tarda (augmentación + extracción de características, y después
el entrenamiento en sí).

In [ ]:
# --overwrite fuerza a recalcular las features aunque ya existan .npy de un intento
# anterior (si no, train.py los reusa tal cual y un reintento después de arreglar un
# bug no vuelve a generar nada — "Openwakeword features already exist, skipping...").
!python3 openWakeWord/openwakeword/train.py --training_config oye_deepy.yaml --augment_clips --overwrite

In [ ]:
!python3 openWakeWord/openwakeword/train.py --training_config oye_deepy.yaml --train_model --overwrite

## 13. Verificar y descargar el modelo

`train.py` a veces exporta el ONNX con los pesos en un archivo externo separado
(`oye_deepy.onnx.data`, ~200 KB) en vez de todo junto en el `.onnx` — si existe, hay que
descargar los dos archivos y guardarlos juntos, en la misma carpeta.

El error final que puede aparecer sobre `onnx_tf`/tflite no importa: es un paso opcional
que `train.py` intenta ejecutar por un bug propio (su flag `--convert_to_tflite` tiene un
valor por defecto que Python evalúa como verdadero aunque nunca se pida), pero ocurre
DESPUÉS de que el `.onnx` ya se exportó bien — no impide que el modelo quede listo.

In [ ]:
modelo_path = f"my_custom_model/{MODEL_NAME}.onnx"
data_path = modelo_path + ".data"
assert os.path.exists(modelo_path), "No se generó el .onnx — revisa el log del paso 12"

import onnxruntime as ort
sess = ort.InferenceSession(modelo_path)
print("Modelo cargado sin errores. Input:", sess.get_inputs()[0].shape)

from google.colab import files
files.download(modelo_path)
if os.path.exists(data_path):
    print("También hay pesos externos, descargando", data_path)
    files.download(data_path)

**Siguiente paso:** guarda el o los archivos descargados (`oye_deepy.onnx`, y
`oye_deepy.onnx.data` si se descargó) en

```
voz_local/voces/wakeword/oye_deepy.onnx
```

en tu PC, y avísale a Claude que ya lo tienes listo para integrarlo en `asistente.py`
(reemplaza el disparo por tecla F9 por la detección de la wake word).